# SharkByte Colab: connected 40-qubit, 33,554,432-state experiment

This is a **new, not-yet-run experiment**. It combines a connected 40-qubit GHZ circuit with 33,554,432 exact sparse amplitudes. The GHZ chain is globally entangled across every nontrivial bipartition; 24 local Hadamards expand the sparse support without changing its Schmidt coefficients.

The worker compares every sparse amplitude to a closed-form formula and runs an independent Cirq Clifford measurement check. This is a structured analytic check, not an independent dense 40-qubit statevector simulation. A dense complex128 vector for 2^40 basis states would require 16 TiB and is not allocated.

Use a **CPU** runtime. The notebook refuses to start below 9 GiB available RAM. The worker runs in a separate process with a 7 GiB address-space limit, a 6.5 GiB sampled RSS stop, a 480-second CPU limit, and a 600-second parent timeout. The parent kills the worker process group on timeout, resource stop, or notebook interruption. Only a compact receipt, log, progress file, and run status are downloaded.

A `PASS` here would establish this exact connected GHZ-family workload; it would not establish dense 40-qubit general-circuit simulation. A resource refusal or worker failure is preserved as such, never reported as a pass.

In [ ]:
import hashlib
import json
import os
from pathlib import Path
import platform
import signal
import subprocess
import sys
import tempfile
import time
import zipfile

GIB = 1024 ** 3
TARGET_STATES = 1 << 25
MIN_AVAILABLE_BYTES = 9 * GIB
WORKER_AS_LIMIT_BYTES = 7 * GIB
WORKER_RSS_LIMIT_BYTES = int(6.5 * GIB)
RUN_DIR = Path(tempfile.mkdtemp(prefix="sharkbyte-connected-40q-", dir="/content"))
REPO = RUN_DIR / "SharkByte"
RESULTS = RUN_DIR / "results"
RESULTS.mkdir()
PACKAGES = RUN_DIR / "python-packages"
PACKAGES.mkdir()
RECEIPT = RESULTS / "sharkbyte-connected-ghz40-33554432-colab.json"
LOG = RESULTS / "worker.log"
PROGRESS = RESULTS / "progress.jsonl"

if not sys.platform.startswith("linux"):
    raise RuntimeError("This guarded notebook requires a Linux CPU runtime.")
clone = subprocess.run(["git", "clone", "--quiet", "--depth", "1", "--branch", "main",
                        "https://github.com/reggaesharkk/SharkByte.git", str(REPO)],
                       capture_output=True, text=True)
if clone.returncode:
    raise RuntimeError(clone.stderr or "GitHub clone failed")
SOURCE_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()

PACKAGES.mkdir(exist_ok=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                "--no-warn-script-location", "--target", str(PACKAGES),
                "numpy==2.5.4", "cirq-core==1.6.1"], check=True)
WORKER_ENV = dict(os.environ, MPLBACKEND="Agg")
old_pythonpath = WORKER_ENV.get("PYTHONPATH", "")
WORKER_ENV["PYTHONPATH"] = str(PACKAGES) + (os.pathsep + old_pythonpath if old_pythonpath else "")
versions = subprocess.check_output([sys.executable, "-c",
    "import cirq,numpy,sys; print(sys.version.split()[0], numpy.__version__, cirq.__version__)"],
    cwd=REPO, env=WORKER_ENV, text=True).strip()
print("SharkByte commit:", SOURCE_COMMIT)
print("Python / NumPy / Cirq:", versions)
print("Target sparse states:", f"{TARGET_STATES:,}")
print("Runtime:", platform.platform())

In [ ]:
def available_memory_bytes():
    meminfo = Path("/proc/meminfo").read_text(encoding="utf-8")
    available = int(next(row.split()[1] for row in meminfo.splitlines() if row.startswith("MemAvailable:"))) * 1024
    for limit_path, usage_path in (("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory.current"),
                                   ("/sys/fs/cgroup/memory/memory.limit_in_bytes",
                                    "/sys/fs/cgroup/memory/memory.usage_in_bytes")):
        try:
            limit_text = Path(limit_path).read_text(encoding="utf-8").strip()
            if limit_text.isdecimal():
                usage = int(Path(usage_path).read_text(encoding="utf-8"))
                available = min(available, max(0, int(limit_text) - usage))
        except FileNotFoundError:
            pass
    return available

def worker_rss_bytes(pid):
    try:
        status = Path(f"/proc/{pid}/status").read_text(encoding="utf-8")
        return int(next(row.split()[1] for row in status.splitlines() if row.startswith("VmRSS:"))) * 1024
    except (FileNotFoundError, StopIteration, ProcessLookupError):
        return 0

def terminate_worker(proc):
    if proc.poll() is not None:
        return
    try:
        os.killpg(proc.pid, signal.SIGTERM)
    except ProcessLookupError:
        return
    try:
        proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        try:
            os.killpg(proc.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        proc.wait()

available_start = available_memory_bytes()
print(f"Available RAM before simulation: {available_start / GIB:.2f} GiB")
print(f"Worker address-space limit: {WORKER_AS_LIMIT_BYTES / GIB:.1f} GiB")
outcome = None
peak_rss = 0
if available_start < MIN_AVAILABLE_BYTES:
    outcome = {"status": "REFUSED", "reason": "Available RAM is below the 9 GiB preflight threshold.",
               "available_bytes": available_start, "required_available_bytes": MIN_AVAILABLE_BYTES}
    (RESULTS / "run-status.json").write_text(json.dumps(outcome, indent=2) + "\n", encoding="utf-8")
    print(outcome)
else:
    def set_worker_limits():
        import resource
        resource.setrlimit(resource.RLIMIT_AS, (WORKER_AS_LIMIT_BYTES, WORKER_AS_LIMIT_BYTES))
        resource.setrlimit(resource.RLIMIT_CPU, (480, 480))
        resource.setrlimit(resource.RLIMIT_CORE, (0, 0))

    started = time.monotonic()
    with LOG.open("w", encoding="utf-8", newline="\n") as log_stream:
        proc = subprocess.Popen([sys.executable, "-B", "tools/run_connected_ghz40_high_support.py",
                                 "--output", str(RECEIPT), "--progress-file", str(PROGRESS),
                                 "--max-seconds", "300"], cwd=REPO, env=WORKER_ENV,
                                stdout=log_stream, stderr=subprocess.STDOUT, start_new_session=True,
                                preexec_fn=set_worker_limits)
        try:
            last_report = 0.0
            while proc.poll() is None:
                peak_rss = max(peak_rss, worker_rss_bytes(proc.pid))
                available_now = available_memory_bytes()
                elapsed = time.monotonic() - started
                if peak_rss > WORKER_RSS_LIMIT_BYTES:
                    outcome = {"status": "RESOURCE_STOP", "reason": "Worker RSS exceeded 6.5 GiB."}
                    terminate_worker(proc)
                    break
                if available_now < 2 * GIB:
                    outcome = {"status": "RESOURCE_STOP", "reason": "Available RAM fell below 2 GiB."}
                    terminate_worker(proc)
                    break
                if elapsed > 600:
                    outcome = {"status": "TIMEOUT", "reason": "600-second parent timeout reached."}
                    terminate_worker(proc)
                    break
                if elapsed - last_report >= 10:
                    latest = PROGRESS.read_text(encoding="utf-8").splitlines()[-1] if PROGRESS.exists() else "starting"
                    print(f"Elapsed {elapsed:.0f}s · worker RSS {peak_rss / GIB:.2f} GiB · {latest}")
                    last_report = elapsed
                time.sleep(1)
        except KeyboardInterrupt:
            terminate_worker(proc)
            outcome = {"status": "CANCELLED", "reason": "Notebook interrupt received; worker process group terminated."}
        except BaseException:
            terminate_worker(proc)
            raise
        return_code = proc.poll()
    parent_elapsed = time.monotonic() - started
    if outcome is None and RECEIPT.exists():
        try:
            record = json.loads(RECEIPT.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError) as exc:
            outcome = {"status": "WORKER_FAILED", "reason": f"Worker receipt is unreadable: {exc}",
                       "return_code": return_code, "sampled_peak_worker_rss_bytes": peak_rss}
        else:
            record["colab_measurement"] = {"available_ram_at_start_bytes": available_start,
                "sampled_peak_worker_rss_bytes": peak_rss, "parent_elapsed_seconds": parent_elapsed,
                "worker_address_space_limit_bytes": WORKER_AS_LIMIT_BYTES,
                "worker_sampled_rss_stop_bytes": WORKER_RSS_LIMIT_BYTES}
            record.pop("receipt_sha256", None)
            canonical = json.dumps(record, sort_keys=True, separators=(",", ":"), allow_nan=False).encode("utf-8")
            record["receipt_sha256"] = hashlib.sha256(canonical).hexdigest()
            RECEIPT.write_text(json.dumps(record, indent=2, sort_keys=True, allow_nan=False) + "\n",
                               encoding="utf-8", newline="\n")
            receipt_file_sha = hashlib.sha256(RECEIPT.read_bytes()).hexdigest()
            if return_code == 0 and record.get("status") == "PASS":
                outcome = {"status": "PASS", "receipt_sha256": receipt_file_sha,
                           "sampled_peak_worker_rss_bytes": peak_rss}
            else:
                outcome = {"status": record.get("status", "WORKER_FAILED"),
                           "reason": f"Worker exited with code {return_code}.",
                           "receipt_sha256": receipt_file_sha,
                           "sampled_peak_worker_rss_bytes": peak_rss}
    elif outcome is None:
        outcome = {"status": "WORKER_FAILED", "reason": f"Worker exited with code {return_code} without a receipt.",
                   "return_code": return_code, "sampled_peak_worker_rss_bytes": peak_rss}
    (RESULTS / "run-status.json").write_text(json.dumps(outcome, indent=2) + "\n", encoding="utf-8")
    print("Run outcome:", json.dumps(outcome))
    if LOG.exists():
        print("\n".join(LOG.read_text(encoding="utf-8", errors="replace").splitlines()[-30:]))
    if PROGRESS.exists():
        print("Last progress:", PROGRESS.read_text(encoding="utf-8").splitlines()[-1])

In [ ]:
archive = RUN_DIR / "sharkbyte-connected-ghz40-32m-results.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(RESULTS.iterdir()):
        if path.is_file():
            bundle.write(path, arcname=path.name)
print("Evidence bundle:", archive)
from google.colab import files
files.download(str(archive))